In [1]:
import polars as pl

In [2]:
books_df = pl.scan_ndjson("../processed-data/cleaned_books_fantasy_paranormal.json")
interactions_df = pl.scan_ndjson("../processed-data/cleaned_interactions_fantasy_paranormal.json")
authors_df = pl.scan_ndjson("../raw-data/goodreads_book_authors.json")
series_df = pl.scan_ndjson("../raw-data/goodreads_book_series.json")

# Ensure join keys have consistent dtypes (some GoodReads dumps mix int/str ids)
books_df = books_df.with_columns(
    pl.col("book_id").cast(pl.Int64, strict=False),
    pl.col("work_id").cast(pl.Int64, strict=False),
    pl.col("ratings_count").cast(pl.Int64, strict=False),
)
interactions_df = interactions_df.with_columns(
    pl.col("book_id").cast(pl.Int64, strict=False)
)

In [3]:
# filter out descriptions with fewer than 50 words
books_df = books_df.filter(pl.col('description').str.count_matches(r"\w+") >= 50)

In [4]:
books_df.select('language_code').unique().show()

language_code
str
""""""
"""swe"""
"""ukr"""
"""vie"""
"""ben"""


In [5]:
# Keep only english variations
# """en"""
# """en-CA"""
# """en-GB"""
# """en-US"""
# """eng"""

books_df = books_df.filter(
    pl.col('language_code').is_in(['en', 'en-CA', 'en-GB', 'en-US', 'eng'])
)


In [6]:
# Deduplicate books by work_id (keep a single representative edition per work).
# Heuristic: prefer higher ratings_count, then longer description, then lower book_id for determinism.
books_df = (
    books_df
    .with_columns(
        pl.col("description").fill_null("").str.len_chars().alias("_desc_len")
    )
    .sort(["work_id", "ratings_count", "_desc_len", "book_id"], descending=[False, True, True, False])
    .unique(subset=["work_id"], keep="first")
    .drop(["_desc_len"])
)

In [7]:
books_df = books_df.explode("authors").with_columns(
    pl.col("authors").struct.field("author_id").alias("author_id")
)
books_df = books_df.join(
    authors_df.select(["author_id", "name"]),
    on="author_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["authors", "author_id", "name"]).first(),
    pl.col("name").drop_nulls().alias("author_names")
)


In [8]:
books_df = books_df.explode("series").with_columns(
    pl.col("series").alias("series_id")
)
books_df = books_df.join(
    series_df.select(["series_id", pl.col("title").alias("series_title")]),
    on="series_id",
    how="left"
).group_by("book_id").agg(
    pl.all().exclude(["series", "series_id", "series_title"]).first(),
    pl.col("series_title").drop_nulls().alias("series")
)


In [9]:
books_df = books_df.with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().struct.field("name")
    ).alias("popular_shelves")
).with_columns(
    pl.col("popular_shelves").list.eval(
        pl.element().filter(
            ~pl.element().str.contains(r"(?i)read|own|buy|fav|library|audio|kindle|ebook")
        )
    )
)


In [10]:
books_df.select('language_code').unique().show()

language_code
str
"""en-GB"""
"""eng"""
"""en-US"""
"""en-CA"""
"""en"""


In [11]:
books_df.head().show()

book_id,isbn,text_reviews_count,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,ratings_count,work_id,title,title_without_series,author_names,series
i64,str,i64,str,str,list[str],str,str,f64,str,list[str],str,str,str,str,i64,i64,str,i64,str,i64,str,str,i64,i64,str,str,list[str],list[str]
3227063,"""0316033677""",3802,"""US""","""en-US""","[""fantasy"", ""fiction"", … ""tbr-pile""]","""""","""false""",4.15,"""B001E0V112""","[""11100431"", ""3428935"", … ""944073""]","""From New York TimesBestselling…","""Mass Market Paperback""","""https://www.goodreads.com/book…","""Orbit""",645,1,"""9780316033671""",10,"""""",2008,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",111914,3261241,"""The Way of Shadows (Night Ange…","""The Way of Shadows (Night Ange…","[""Brent Weeks""]","[""Night Angel""]"
31212883,"""1626399131""",5,"""US""","""eng""","[""lgbt"", ""fantasy"", … ""lgbtq""]","""""","""false""",3.96,"""B06XPF6GN2""",[],"""The Iron Phoenix, the masked v…","""Paperback""","""https://www.goodreads.com/book…","""Bold Strokes Books""",240,18,"""9781626399136""",4,"""""",2017,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",21,51867495,"""Phoenix Rising (Storm's Quarry…","""Phoenix Rising (Storm's Quarry…","[""Rebecca Harwell""]","[""Storm's Quarry""]"
9325,"""1421504618""",108,"""US""","""eng""","[""manga"", ""mangá"", … ""my-books""]","""""","""false""",4.6,"""B00JDRL1WM""","[""294971"", ""1792371"", … ""13621""]","""In an alchemical ritual gone w…","""Paperback""","""https://www.goodreads.com/book…","""VIZ Media LLC""",200,21,"""9781421504612""",11,"""""",2006,"""https://www.goodreads.com/book…","""https://s.gr-assets.com/assets…",7147,3001807,"""Fullmetal Alchemist, Vol. 10 (…","""Fullmetal Alchemist, Vol. 10 (…","[""Akira Watanabe"", ""Hiromu Arakawa""]","[""Fullmetal Alchemist""]"
13333340,"""192050253X""",14,"""US""","""eng""","[""paranormal"", ""m-m"", … ""injury-illness""]","""""","""true""",3.44,"""""","[""15983583"", ""13598107"", … ""15829313""]","""With renovations of his old Go…","""ebook""","""https://www.goodreads.com/book…","""Silver Publishing""",175,31,"""9781920502539""",3,"""1st Edition""",2012,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",151,18541275,"""Stone Heart""","""Stone Heart""","[""Sui Lynn""]","[""Stone Hearts Serial""]"
29350632,"""""",10,"""US""","""eng""","[""fantasy"", ""giveaways"", … ""giveaways-entered""]","""""","""false""",4.54,"""""","[""463367"", ""28600081"", … ""29856826""]","""Rulla, Dealer of Fates, has se…","""Paperback""","""https://www.goodreads.com/book…","""McKinley Writing""",333,26,"""9780993880223""",4,"""""",2016,"""https://www.goodreads.com/book…","""https://images.gr-assets.com/b…",13,49590079,"""Harbinger (Northern Fire #1)""","""Harbinger (Northern Fire #1)""","[""Ian H. McKinley""]","[""Northern Fire""]"


In [12]:
books_df = books_df.drop([
    'isbn', 'text_reviews_count', 'country_code', 'language_code',
    'is_ebook', 'kindle_asin', 'format', 'num_pages',
    'publication_day', 'isbn13', 'publication_month', 'edition_information', 'publication_year', 'image_url',
    'title_without_series', 'ratings_count', 'publisher', 'similar_books', 'asin'
])
books_df.head().show()


book_id,popular_shelves,average_rating,description,link,url,work_id,title,author_names,series
i64,list[str],f64,str,str,str,i64,str,list[str],list[str]
25062765,"[""paranormal"", ""romance"", … ""fiction-paranormal""]",3.94,"""Art appraiser Addison Montgome…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",44167447,"""Undoing Time (The Fine Art of …","[""Alyssa Richards""]","[""The Fine Art of Deception""]"
24244681,"[""fantasy"", ""young-adult"", … ""middle-grade-fiction""]",3.93,"""A monstrously funny debut from…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",42391550,"""Darkmouth (Darkmouth, Book 1)""","[""Shane Hegarty""]","[""Darkmouth""]"
5201659,"[""fantasy"", ""urban-fantasy"", … ""sequels""]",3.99,"""For seeker Raine Benares, a de…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",5268814,"""The Trouble with Demons (Raine…","[""Lisa Shearin""]","[""Raine Benares""]"
27386671,"[""m-m"", ""paranormal"", … ""anthologies""]",3.4,"""This is the complete Love Amon…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",47430672,"""Love Among The Vampires - Comp…","[""Aiden Bates""]","[""Love Among the Vampires""]"
17374789,"[""fantasy"", ""magic"", … ""look-for-sequel""]",3.6,"""Cold. Dry. Dead. That's all I …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",24163374,"""Obsidian (Ember, #2)""","[""Tess Williams""]","[""Ember""]"


In [13]:
books_df = books_df.with_columns(
    pl.concat_str(
        [
            pl.col("title").fill_null(""),
            pl.col("title").fill_null(""),
            pl.col("title").fill_null(""),
            pl.col("series").list.join(", ").fill_null(""),
            pl.col("popular_shelves").list.join(", ").fill_null(""),
            pl.col("description").fill_null("")
        ],
        separator=" "
    ).alias("combined_text")
)


In [14]:
# Basic text preprocessing on combined_text
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOPWORDS = set(stopwords.words('english'))
stopwords_regex = r"(?i)\b(" + "|".join(STOPWORDS) + r")\b"

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    if not text:
        return ""
    return " ".join([lemmatizer.lemmatize(w) for w in text.split()])

books_df = books_df.with_columns(
    pl.col("combined_text")
    .str.to_lowercase()
    .str.replace_all(r"[^\w\s]", " ") # remove punctuation
    .str.replace_all(stopwords_regex, "") # remove stopwords
    .str.replace_all(r"\s+", " ") # remove multiple spaces
    .str.strip_chars() # trim leading/trailing spaces
    .map_elements(lemmatize_text, return_dtype=pl.String)
)

books_df.head().show()


book_id,popular_shelves,average_rating,description,link,url,work_id,title,author_names,series,combined_text
i64,list[str],f64,str,str,str,i64,str,list[str],list[str],str
16357873,"[""fantasy"", ""one-a-day-4"", … ""2013-willow-nominees""]",4.23,"""Short-listed for the Hackmatac…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",22512540,"""The Gargoyle at the Gates""","[""Philippa Dowding""]",[],"""gargoyle gate gargoyle gate ga…"
25746600,"[""romance"", ""young-adult"", … ""supes""]",3.53,"""Finding love is hard, even whe…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",45587720,"""Love, Lattes and Mutants""","[""Sandra Cox""]","[""Mutants""]","""love latte mutant love latte m…"
20939506,"[""romance"", ""paranormal"", … ""online-dating-smutt""]",4.01,"""Yvena needs to start over and …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",40310131,"""Snake Charmer (Shifting Crossr…","[""Zenina Masters""]","[""Shifting Crossroads""]","""snake charmer shifting crossro…"
12384961,"[""fantasy"", ""young-adult"", … ""mine""]",3.99,"""While caring for her uncle in …","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",1112036,"""Magic Steps (The Circle Opens,…","[""Tamora Pierce""]","[""Emelan"", ""Emelan Chronological Order"", ""The Circle Opens""]","""magic step circle open 1 magic…"
32218143,"[""vampires"", ""paranormal-fiction"", … ""cozy""]",3.88,"""An alternate cover for this AS…","""https://www.goodreads.com/book…","""https://www.goodreads.com/book…",52858763,"""Murder Bites (The Vampire Myst…","[""Lyra Barnett""]","[""The Vampire Mysteries""]","""murder bite vampire mystery 1 …"


In [15]:
import os
os.makedirs("../processed-data", exist_ok=True)
books_df.collect().write_ndjson("../processed-data/processed_books_texts.json")
print("Dataset successfully saved!")

Dataset successfully saved!
